# Wikipedia Semantics Graph Analysis
This notebook analyzes the properties of the Wikipedia knowledge graph generated by the `generate_edge_list.py` script.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
import time

# Import the load_graph utility from our model training script
from train_model import load_graph

# Configure matplotlib for nicer plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load the graph
edge_list_path = 'data/edge_list.txt'
print(f"Loading graph from {edge_list_path}...")
t0 = time.time()
edges, adj_matrix, all_nodes, orig_to_compact, compact_to_orig = load_graph(edge_list_path)
print(f"Graph loaded in {time.time() - t0:.2f} seconds.")
print(f"Number of nodes: {adj_matrix.shape[0]:,}")
print(f"Number of edges: {adj_matrix.nnz:,}")

## Degree Distribution
Let's analyze the in-degree and out-degree of nodes in the graph. In a scale-free network like Wikipedia, we expect this to follow a power-law distribution.

In [ ]:
# Calculate out-degrees (row sums) and in-degrees (column sums)
out_degrees = np.array(adj_matrix.sum(axis=1)).flatten()
in_degrees = np.array(adj_matrix.sum(axis=0)).flatten()

# Filter out zero degrees for log-log plotting
out_degrees_nz = out_degrees[out_degrees > 0]
in_degrees_nz = in_degrees[in_degrees > 0]

print(f"Max out-degree: {out_degrees.max():,}")
print(f"Max in-degree: {in_degrees.max():,}")
print(f"Mean out-degree: {out_degrees.mean():.2f}")
print(f"Mean in-degree: {in_degrees.mean():.2f}")

# Plot Degree Distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Out-degree histogram
bins = np.logspace(0, np.log10(out_degrees_nz.max()), 50)
ax1.hist(out_degrees_nz, bins=bins, color='skyblue', edgecolor='black', alpha=0.7)
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_title('Out-Degree Distribution (Log-Log)')
ax1.set_xlabel('Out-Degree')
ax1.set_ylabel('Frequency')

# In-degree histogram
bins = np.logspace(0, np.log10(in_degrees_nz.max()), 50)
ax2.hist(in_degrees_nz, bins=bins, color='lightcoral', edgecolor='black', alpha=0.7)
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_title('In-Degree Distribution (Log-Log)')
ax2.set_xlabel('In-Degree')
ax2.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## Graph Connectivity (Connected Components)
We use `scipy.sparse.csgraph` to find the strongly and weakly connected components of the Wikipedia graph.

In [ ]:
# Find Weakly Connected Components (WCC)
print("Computing Weakly Connected Components...")
t0 = time.time()
n_components_weak, labels_weak = connected_components(csgraph=adj_matrix, directed=True, connection='weak')
print(f"Found {n_components_weak:,} weakly connected components in {time.time() - t0:.2f}s.")

# Analyze WCC sizes
unique_weak, counts_weak = np.unique(labels_weak, return_counts=True)
component_sizes_weak = np.sort(counts_weak)[::-1]
print(f"Largest WCC size: {component_sizes_weak[0]:,} nodes ({(component_sizes_weak[0] / adj_matrix.shape[0]) * 100:.2f}% of graph)")

# Find Strongly Connected Components (SCC)
print("\nComputing Strongly Connected Components...")
t0 = time.time()
n_components_strong, labels_strong = connected_components(csgraph=adj_matrix, directed=True, connection='strong')
print(f"Found {n_components_strong:,} strongly connected components in {time.time() - t0:.2f}s.")

# Analyze SCC sizes
unique_strong, counts_strong = np.unique(labels_strong, return_counts=True)
component_sizes_strong = np.sort(counts_strong)[::-1]
print(f"Largest SCC size: {component_sizes_strong[0]:,} nodes ({(component_sizes_strong[0] / adj_matrix.shape[0]) * 100:.2f}% of graph)")

In [ ]:
# Plot Component Size Distribution
plt.figure(figsize=(10, 6))

# Only plot top 100 components to avoid clutter if there are many tiny ones
top_k = min(100, len(component_sizes_strong))

plt.plot(range(1, top_k + 1), component_sizes_strong[:top_k], marker='o', linestyle='-', color='purple', label='Strongly Connected Components')
plt.plot(range(1, min(100, len(component_sizes_weak)) + 1), component_sizes_weak[:min(100, len(component_sizes_weak))], marker='s', linestyle='--', color='green', label='Weakly Connected Components')

plt.yscale('log')
plt.xscale('log')
plt.title('Component Size vs. Rank (Top 100)')
plt.xlabel('Component Rank')
plt.ylabel('Number of Nodes in Component')
plt.legend()
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.show()